In [0]:
%run ./00_config

## テーブル設計書: Customer Data Platform (Travel)

| # | テーブル名 | PK | FK | 概要 |
| --- | --- | --- | --- | --- |
| 1 | customers | user_id | — | 旅行者プロファイル（Profile Entity） |
| 2 | flight_activity | flight_id | — | フライト運航情報 |
| 3 | app_sessions | event_id | user_id → customers | アプリ/Web/Push イベント |
| 4 | bookings | booking_id | user_id → customers, flight_id → flight_activity | 予約情報 |
| 5 | customer_support_tickets | ticket_id | user_id → customers | サポートチケット |
| 6 | ancillary_purchases | purchase_id | user_id → customers, booking_id → bookings | 付帯購入 |

> **用途**: 別ワークスペースでこれらのテーブルを再現するための DDL 設計書。  
> PK/FK は Databricks の informational constraint（非強制）として定義。  
> カタログ名・スキーマ名は移行先に合わせて置換してください。

## 1. 空テーブルの作成

In [0]:
%sql
CREATE TABLE IF NOT EXISTS customers (
  user_id BIGINT COMMENT '旅行者ID（主キー）',
  first_name STRING COMMENT '名' /* PII */,
  last_name STRING COMMENT '姓' /* PII */,
  email STRING COMMENT 'メールアドレス' /* PII */,
  phone STRING COMMENT '電話番号' /* PII */,
  country_code STRING COMMENT '居住国コード',
  home_airport STRING COMMENT '主要出発空港',
  preferred_language STRING COMMENT 'コミュニケーション言語',
  travel_purpose STRING COMMENT '旅行目的（ビジネス/レジャー/混合）',
  region_focus STRING COMMENT '地域志向（国内/国際/両方）',
  loyalty_id STRING COMMENT 'マイレージ番号',
  loyalty_tier STRING COMMENT '会員ランク（Member/Silver/Gold/Platinum）',
  tier_status STRING COMMENT 'ランクステータス（Active/grace/lapsed）',
  points_balance BIGINT COMMENT '現在の交換可能マイル数',
  annual_rewards_miles INT COMMENT '今年獲得したマイル数',
  marketing_consent_status STRING COMMENT 'マーケティング同意状況',
  preferred_channel STRING COMMENT '希望連絡チャネル（Email/Push/SMS）',
  mobile_app_installed BOOLEAN COMMENT 'アプリインストール済み',
  co_branded_card_holder BOOLEAN COMMENT '提携カード保有者',
  churn_risk_score FLOAT COMMENT '離反リスクスコア',
  source_systems ARRAY<STRING> COMMENT 'ソースシステム（CRM, ロイヤルティ, 予約, アプリ, Web）',
  identity_resolution_status STRING COMMENT 'ID統合ステータス（Resolved/Partial）',
  created_at TIMESTAMP COMMENT 'プロファイル作成日時',
  updated_at TIMESTAMP COMMENT '最終更新日時',
  CONSTRAINT pk_customers PRIMARY KEY (user_id)
)
COMMENT 'プロファイルエンティティ - ロイヤルティ、嗜好、離反スコアリングを含む旅行者マスタ';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS flight_activity (
  flight_id BIGINT COMMENT 'フライトID',
  flight_number STRING COMMENT '航空会社コード＋便名',
  origin_airport STRING COMMENT '出発空港',
  destination_airport STRING COMMENT '到着空港',
  route STRING COMMENT '路線（出発-到着ペア）',
  flight_date DATE COMMENT '運航日',
  aircraft_type STRING COMMENT '機材タイプ',
  seat_capacity INT COMMENT '販売可能座席数',
  load_factor FLOAT COMMENT '搭乗率（予約数÷座席数）',
  delay_minutes INT COMMENT '出発遅延（分）',
  flight_status STRING COMMENT '運航状況（Scheduled/flown/cancelled）',
  CONSTRAINT pk_flight_activity PRIMARY KEY (flight_id)
)
COMMENT 'フライト運航記録（座席数、搭乗率、遅延情報を含む）';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS app_sessions (
  event_id STRING COMMENT 'イベントID',
  user_id BIGINT COMMENT '旅行者への外部キー',
  event_timestamp TIMESTAMP COMMENT 'イベント発生日時',
  event_type STRING COMMENT 'イベント種別（search/fare_view/booking_start/email_open/push_click）',
  channel STRING COMMENT 'チャネル（Web/App/Email/Push/Paid）',
  origin_airport STRING COMMENT '検索された出発空港',
  destination_airport STRING COMMENT '検索された到着空港',
  campaign_id STRING COMMENT 'キャンペーンID（任意）',
  metadata STRING COMMENT 'イベント詳細（JSON形式）',
  CONSTRAINT pk_app_sessions PRIMARY KEY (event_id),
  CONSTRAINT fk_app_sessions_user FOREIGN KEY (user_id) REFERENCES customers (user_id)
)
COMMENT '行動分析およびアトリビューション用のアプリ/Web/Pushイベントストリーム';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bookings (
  booking_id BIGINT COMMENT '予約ID',
  user_id BIGINT COMMENT '旅行者への外部キー',
  flight_id BIGINT COMMENT '往路フライトID',
  record_locator STRING COMMENT '予約確認番号',
  trip_type STRING COMMENT '旅程タイプ（片道/往復/周遊）',
  origin_airport STRING COMMENT '出発地',
  destination_airport STRING COMMENT '目的地',
  fare_class STRING COMMENT '運賃クラス',
  cabin STRING COMMENT '予約キャビン',
  booking_channel STRING COMMENT '予約チャネル（App/Web/コールセンター/パートナー）',
  total_fare FLOAT COMMENT '支払運賃',
  booked_at TIMESTAMP COMMENT '予約日時',
  CONSTRAINT pk_bookings PRIMARY KEY (booking_id),
  CONSTRAINT fk_bookings_user FOREIGN KEY (user_id) REFERENCES customers (user_id),
  CONSTRAINT fk_bookings_flight FOREIGN KEY (flight_id) REFERENCES flight_activity (flight_id)
)
COMMENT '旅行者とフライトを紐づける予約トランザクション';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS customer_support_tickets (
  ticket_id BIGINT COMMENT 'サポートチケットID',
  user_id BIGINT COMMENT '旅行者への外部キー',
  case_type STRING COMMENT '問い合わせ種別（返金/運航障害/手荷物/ロイヤルティ/一般）',
  status STRING COMMENT 'ステータス（Open/pending/escalated/resolved/closed）',
  priority STRING COMMENT '優先度（Low/medium/high/urgent）',
  channel STRING COMMENT '問い合わせチャネル（電話/チャット/メール/空港/アプリ）',
  opened_at TIMESTAMP COMMENT 'チケット作成日時',
  resolved_at TIMESTAMP COMMENT '解決日時',
  last_updated_at TIMESTAMP COMMENT '最終更新日時',
  CONSTRAINT pk_support_tickets PRIMARY KEY (ticket_id),
  CONSTRAINT fk_support_tickets_user FOREIGN KEY (user_id) REFERENCES customers (user_id)
)
COMMENT 'ステータス追跡と解決日時を含む顧客サポートケース';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ancillary_purchases (
  purchase_id BIGINT COMMENT '購入ID',
  user_id BIGINT COMMENT '旅行者への外部キー',
  booking_id BIGINT COMMENT '予約への外部キー',
  purchase_type STRING COMMENT '購入種別（座席/手荷物/ラウンジ/アップグレード/Wi-Fi）',
  product_name STRING COMMENT '付帯商品名',
  quantity INT COMMENT '購入数量',
  revenue FLOAT COMMENT '購入金額',
  purchase_channel STRING COMMENT '購入チャネル（Web/アプリ/キオスク/機内）',
  purchase_date DATE COMMENT '購入日',
  refunded_flag BOOLEAN COMMENT '返金済みフラグ',
  CONSTRAINT pk_ancillary_purchases PRIMARY KEY (purchase_id),
  CONSTRAINT fk_ancillary_user FOREIGN KEY (user_id) REFERENCES customers (user_id),
  CONSTRAINT fk_ancillary_booking FOREIGN KEY (booking_id) REFERENCES bookings (booking_id)
)
COMMENT '予約および旅行者に紐づく付帯収益アイテム';

## 2. 空テーブルにデータ投入

In [0]:
from pyspark.sql.types import *
import os

# CSV データのパス（このノートブックと同階層の _data/ ディレクトリ）
notebook_dir = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
data_dir = f"/Workspace{notebook_dir}/_data"

tables = [
    "customers",
    "flight_activity",
    "bookings",
    "ancillary_purchases",
    "app_sessions",
    "customer_support_tickets",
]

for table_name in tables:
    csv_path = f"{data_dir}/{table_name}.csv.gz"
    df = spark.read.csv(csv_path, header=True, inferSchema=True)

    # ARRAY型カラムの変換（CSVは文字列として読み込まれるため）
    from pyspark.sql.types import ArrayType
    from pyspark.sql import functions as F
    target_schema = spark.table(table_name).schema
    for field in target_schema:
        if isinstance(field.dataType, ArrayType) and field.name in df.columns:
            df = df.withColumn(field.name,
                F.from_json(F.regexp_replace(F.col(field.name), "'", '"'), field.dataType))

    df.createOrReplaceTempView(f"_tmp_{table_name}")
    spark.sql(f"INSERT OVERWRITE TABLE {table_name} SELECT * FROM _tmp_{table_name}")
    print(f"✅ {table_name}: {spark.table(table_name).count():,} rows")

print("\n=== 全テーブルデータ取込完了 ===")